# Ames Housing: Feature Engineering & Exploratory Data Analysis

A structured analysis of residential properties in Ames, Iowa, with a focus on understanding the variables associated with sale prices and preparing a clean dataset for later modelling.

**Project type:** Data Cleaning · Feature Analysis · Exploratory Data Analysis  
**Target column:** `SalePrice`  
**Dataset file:** `PEP1.csv`

> This notebook follows the supplied Feature Engineering assignment. It explores and prepares the data; it does not train or evaluate a predictive model.

## 1. Project Objectives

The aim is to inspect the housing data, understand its numerical and categorical variables, and prepare a consistent dataset for future analysis.

The notebook covers:

- Loading the CSV and checking its structure.
- Reviewing column names, dimensions, data types, unique values, and missing data.
- Separating numerical and categorical features.
- Exploring distributions and categorical frequencies.
- Treating missing values using simple statistical imputations.
- Measuring skewness and examining feature correlations with `SalePrice`.
- Using pair plots and box plots to inspect relationships and potential outliers.
- Saving a cleaned copy of the dataset.

The dataset description refers to 79 explanatory variables describing residential homes in Ames, Iowa. The actual row and column counts are read from the CSV rather than assumed.

## 2. Libraries and Display Settings

The libraries below cover data handling, numerical operations, and visual analysis. A consistent plotting style keeps the charts readable when the notebook is viewed on GitHub.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

sns.set_theme(style="whitegrid", context="notebook")

## 3. Load the Dataset

On Kaggle, the file is expected at the dataset input path shown below. For local use, place `PEP1.csv` in a `data` directory beside the notebook, or update `DATA_PATH` to point to its location.

The code checks a few common locations and stops with a clear message if the file cannot be found.

In [ ]:
possible_paths = [
    Path("/kaggle/input/datasets/harikantshukla/feature-engineering-dataset/PEP1.csv"),
    Path("data/PEP1.csv"),
    Path("PEP1.csv"),
]

DATA_PATH = next((path for path in possible_paths if path.exists()), None)

if DATA_PATH is None:
    checked = "\\n".join(f"- {path}" for path in possible_paths)
    raise FileNotFoundError(
        "PEP1.csv was not found. Place it in the data folder or update DATA_PATH. "
        f"\\nLocations checked:\\n{checked}"
    )

df = pd.read_csv(DATA_PATH)

print(f"Loaded: {DATA_PATH}")
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
display(df.head())

## 4. First Look at the Data

Start with a few rows, the data types, and descriptive statistics. These checks help distinguish numeric measurements from text-based categories and reveal unusual ranges that may need a closer look.

In [ ]:
print("First five records")
display(df.head())

print("Last five records")
display(df.tail())

print("Data types and non-null counts")
df.info()

print("Descriptive statistics")
display(df.describe(include="all").T)

## 5. Column Names and Dataset Dimensions

The assignment describes 79 explanatory variables. The target is expected to be `SalePrice`. Confirm the actual schema here before proceeding; dataset copies can differ in their exact columns.

In [ ]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
print("\\nColumn names:")
for column in df.columns:
    print(column)

TARGET = "SalePrice"

if TARGET not in df.columns:
    raise KeyError(
        f"The expected target column '{TARGET}' is missing. "
        "Check the CSV schema and update TARGET if the file uses another name."
    )

feature_columns = [column for column in df.columns if column != TARGET]
print(f"\\nExplanatory columns: {len(feature_columns)}")
print(f"Target column: {TARGET}")

## 6. Missing-Value Assessment

Missing values are common in housing data. A blank value can mean that information was not recorded, or that a feature does not apply to a particular property. Those meanings are not always interchangeable, so review the missingness before choosing an imputation strategy.

In [ ]:
missing_count = df.isna().sum()
missing_percent = (df.isna().mean() * 100).round(2)

missing_summary = (
    pd.DataFrame({
        "missing_count": missing_count,
        "missing_percent": missing_percent,
    })
    .query("missing_count > 0")
    .sort_values("missing_percent", ascending=False)
)

if missing_summary.empty:
    print("No missing values were found.")
else:
    display(missing_summary)

## 7. Unique Values and Feature Types

A quick unique-value count helps identify columns that behave like categories even when they are stored as numbers. The automatic type split below is based on pandas data types; it is a useful starting point, not a substitute for checking the meaning of each column.

In [ ]:
unique_summary = pd.DataFrame({
    "unique_values": df.nunique(dropna=True),
    "dtype": df.dtypes.astype(str),
}).sort_values("unique_values")

display(unique_summary)

for column in df.columns:
    values = df[column].dropna().unique()
    print(f"{column}: {len(values)} unique values")
    if len(values) <= 15:
        print("   ", sorted(values.tolist(), key=str))

numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print(f"\\nNumerical columns ({len(numerical_cols)}):")
print(numerical_cols)

print(f"\\nCategorical columns ({len(categorical_cols)}):")
print(categorical_cols)

## 8. Exploratory Data Analysis

### 8.1 Numerical Feature Distributions

Histograms show the shape and spread of each numerical variable. The optional KDE curve gives a smoothed view of the distribution. Highly skewed features and unusual peaks are worth investigating before selecting transformations.

In [ ]:
if numerical_cols:
    columns_per_row = 3
    rows = int(np.ceil(len(numerical_cols) / columns_per_row))
    fig, axes = plt.subplots(rows, columns_per_row, figsize=(16, 4 * rows))
    axes = np.asarray(axes).reshape(-1)

    for index, column in enumerate(numerical_cols):
        sns.histplot(data=df, x=column, kde=True, ax=axes[index])
        axes[index].set_title(f"Distribution of {column}")

    for axis in axes[len(numerical_cols):]:
        axis.remove()

    fig.tight_layout()
    plt.show()
else:
    print("No numerical columns are available for plotting.")

### 8.2 Categorical Feature Frequencies

Value counts show how observations are distributed across the categories in each text or categorical column. Very high-cardinality columns may be better inspected separately.

In [ ]:
for column in categorical_cols:
    print(f"\\n{column}")
    display(df[column].value_counts(dropna=False).head(20).to_frame("count"))

### 8.3 Missingness by Feature

This chart highlights which columns have the largest proportion of missing observations. It is a visual companion to the missing-value table above.

In [ ]:
if not missing_summary.empty:
    plt.figure(figsize=(11, max(4, 0.35 * len(missing_summary))))
    sns.barplot(
        data=missing_summary.reset_index(names="feature"),
        x="missing_percent",
        y="feature",
    )
    plt.title("Missing Values by Feature")
    plt.xlabel("Missing values (%)")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()
else:
    print("There are no missing values to plot.")

## 9. Missing-Value Treatment

For this exploratory preparation step, numerical feature values are filled with their column median and categorical feature values with their most frequent category.

Rows with a missing `SalePrice` are removed because the target is not suitable for median or mode imputation. This avoids manufacturing a target value.

**Modelling note:** If this prepared data is later split into training and test sets, imputation values should be learned from the training partition only, preferably inside a scikit-learn pipeline. Applying imputation to the full dataset before a split can leak information from the test set.

In [ ]:
df_prepared = df.copy()

# A missing target cannot provide a supervised learning example.
missing_target_rows = df_prepared[TARGET].isna().sum()
if missing_target_rows:
    df_prepared = df_prepared.dropna(subset=[TARGET]).copy()
    print(f"Removed {missing_target_rows} rows with missing {TARGET}.")

# Recalculate feature groups after removing rows with an unknown target.
feature_numerical_cols = [
    column for column in df_prepared.select_dtypes(include=np.number).columns
    if column != TARGET
]
feature_categorical_cols = df_prepared.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

numeric_fill_values = {}
categorical_fill_values = {}

for column in feature_numerical_cols:
    if df_prepared[column].isna().any():
        fill_value = df_prepared[column].median()
        df_prepared[column] = df_prepared[column].fillna(fill_value)
        numeric_fill_values[column] = fill_value

for column in feature_categorical_cols:
    if df_prepared[column].isna().any():
        modes = df_prepared[column].mode(dropna=True)
        if not modes.empty:
            fill_value = modes.iloc[0]
            df_prepared[column] = df_prepared[column].fillna(fill_value)
            categorical_fill_values[column] = fill_value

print("Numerical columns filled:", len(numeric_fill_values))
print("Categorical columns filled:", len(categorical_fill_values))
print("Remaining missing values:", int(df_prepared.isna().sum().sum()))

display(pd.DataFrame({
    "numerical_median": pd.Series(numeric_fill_values),
    "categorical_mode": pd.Series(categorical_fill_values),
}))

## 10. Skewness Analysis

Skewness describes the asymmetry of a distribution. A positive value indicates a longer right tail, while a negative value indicates a longer left tail. Skewness alone does not determine whether a transformation is appropriate; the feature's meaning and the modelling method also matter.

In [ ]:
skew_values = df_prepared[feature_numerical_cols].skew().sort_values(ascending=False)

def describe_skew(value):
    if value > 1:
        return "Highly right-skewed"
    if value > 0.5:
        return "Moderately right-skewed"
    if value < -1:
        return "Highly left-skewed"
    if value < -0.5:
        return "Moderately left-skewed"
    return "Approximately symmetric"

skew_summary = pd.DataFrame({
    "skewness": skew_values,
    "interpretation": skew_values.map(describe_skew),
})

display(skew_summary)

## 11. Correlation with SalePrice

Pearson correlation measures the strength and direction of a linear relationship between two numerical variables. The ranking below is an exploratory signal, not proof that a feature causes a change in sale price. Correlation can also miss non-linear relationships.

In [ ]:
numeric_for_corr = df_prepared.select_dtypes(include=np.number)
corr_matrix = numeric_for_corr.corr()

saleprice_corr = (
    corr_matrix[TARGET]
    .drop(labels=[TARGET])
    .sort_values(key=lambda values: values.abs(), ascending=False)
)

display(saleprice_corr.to_frame("correlation_with_SalePrice"))

plt.figure(figsize=(10, max(5, 0.28 * len(saleprice_corr))))
sns.barplot(
    x=saleprice_corr.values,
    y=saleprice_corr.index,
    hue=saleprice_corr.index,
    palette="coolwarm",
    legend=False,
)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Numerical Feature Correlation with SalePrice")
plt.xlabel("Pearson correlation")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

### 11.1 Correlation Heatmap

A heatmap gives a broader view of relationships among numerical columns. Strong correlations between explanatory variables can indicate multicollinearity, which should be considered when selecting and interpreting a later model.

In [ ]:
plt.figure(figsize=(16, 12))
sns.heatmap(
    corr_matrix,
    cmap="coolwarm",
    center=0,
    linewidths=0.2,
    cbar_kws={"shrink": 0.8},
)
plt.title("Correlation Matrix of Numerical Variables")
plt.tight_layout()
plt.show()

## 12. Pair Plot

A pair plot compares selected numerical variables two at a time and shows each variable's distribution on the diagonal. To keep the figure readable, the notebook selects `SalePrice` and up to five numerical features with the strongest absolute correlation to it.

In [ ]:
selected_pairplot_cols = [TARGET] + saleprice_corr.head(5).index.tolist()
selected_pairplot_cols = list(dict.fromkeys(selected_pairplot_cols))

if len(selected_pairplot_cols) > 1:
    pair_data = df_prepared[selected_pairplot_cols].dropna()
    # Sampling keeps the chart responsive on larger CSV files.
    if len(pair_data) > 1000:
        pair_data = pair_data.sample(1000, random_state=42)

    pair_grid = sns.pairplot(
        pair_data,
        corner=True,
        diag_kind="hist",
        plot_kws={"alpha": 0.55, "s": 18},
    )
    pair_grid.fig.suptitle("Pair Plot: SalePrice and Selected Features", y=1.02)
    plt.show()
else:
    print("Not enough numerical columns for a pair plot.")

## 13. Outlier Inspection with Box Plots

Box plots show the median, interquartile range, and observations beyond the conventional 1.5 × IQR whiskers. Such observations are flagged for inspection; they are not automatically errors and should not be removed without understanding the property or measurement context.

In [ ]:
if numerical_cols:
    plot_cols = [column for column in numerical_cols if column in df_prepared.columns]
    columns_per_row = 3
    rows = int(np.ceil(len(plot_cols) / columns_per_row))
    fig, axes = plt.subplots(rows, columns_per_row, figsize=(16, 4 * rows))
    axes = np.asarray(axes).reshape(-1)

    for index, column in enumerate(plot_cols):
        sns.boxplot(y=df_prepared[column], ax=axes[index])
        axes[index].set_title(f"Box plot: {column}")

    for axis in axes[len(plot_cols):]:
        axis.remove()

    fig.tight_layout()
    plt.show()

### 13.1 IQR Outlier Counts

For each numerical column, the interquartile range is calculated as `Q3 - Q1`. Values below `Q1 - 1.5 × IQR` or above `Q3 + 1.5 × IQR` are counted as potential outliers.

These counts are a screening aid, not a decision to delete records.

In [ ]:
outlier_records = []

for column in numerical_cols:
    values = df_prepared[column].dropna()
    if values.empty:
        continue

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    count = ((values < lower_bound) | (values > upper_bound)).sum()

    outlier_records.append({
        "feature": column,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "outlier_count": int(count),
        "outlier_percent": round(count / len(values) * 100, 2),
    })

outlier_summary = (
    pd.DataFrame(outlier_records)
    .sort_values("outlier_count", ascending=False)
    .reset_index(drop=True)
)

display(outlier_summary)

## 14. Prepare and Save the Cleaned Dataset

The prepared copy retains the original columns, fills missing explanatory values using the rules documented above, and excludes rows without a known target. The source dataframe `df` remains unchanged.

The output is intended as a starting point for further feature engineering and modelling. For a rigorous model evaluation, perform train/test splitting first and fit all learned preprocessing steps using only the training data.

In [ ]:
OUTPUT_PATH = Path("feature_engineering_dataset_cleaned.csv")
df_prepared.to_csv(OUTPUT_PATH, index=False)

print(f"Saved prepared dataset to: {OUTPUT_PATH.resolve()}")
print(f"Prepared shape: {df_prepared.shape[0]:,} rows × {df_prepared.shape[1]:,} columns")
print(f"Missing values remaining: {int(df_prepared.isna().sum().sum())}")
display(df_prepared.head())

## 15. Summary and Next Steps

This notebook has completed the requested data understanding and exploratory preparation tasks:

- Inspected the dataset structure, columns, and data types.
- Reviewed missing values and unique-value counts.
- Separated numerical and categorical variables.
- Examined distributions, category frequencies, and skewness.
- Explored numerical correlations with `SalePrice`.
- Used pair plots and box plots to inspect relationships and potential outliers.
- Created a prepared CSV for later work.

### Possible next steps

1. Review the dataset's `data_description.txt` file to understand the meaning of individual Ames Housing fields and special missing-value conventions.
2. Decide whether missing values represent absence (for example, a feature not present in a house) or genuinely unknown information.
3. Build a preprocessing pipeline that imputes values and encodes categorical variables.
4. Split data into training and test sets before fitting preprocessing transformations.
5. Train and compare regression models in a separate modelling notebook.

### Responsible interpretation

The exploratory correlations and outlier flags describe patterns in this dataset. They do not establish causation or guarantee that the same relationships will hold in another housing market or time period.

## References

- Kaggle: **Feature Engineering Dataset** — use the dataset page associated with the supplied `PEP1.csv` file.
- Ames Housing data documentation: `data_description.txt`, supplied through the original Feature Engineering project problem statement.

The dataset description identifies 79 explanatory variables for residential homes in Ames, Iowa. Consult the accompanying data dictionary before interpreting individual fields or assigning meaning to missing values.